#STEP 1: Import Libraries and Upload Daraset

**1. Install and import libraries.**

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

**2. Upload the data**

In [ ]:
# Download the data

!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz

--2026-03-29 10:06:48--  https://archive.ics.uci.edu/ml/machine-learning-databases/00492/Metro_Interstate_Traffic_Volume.csv.gz
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘Metro_Interstate_Traffic_Volume.csv.gz.1’

Metro_Interstate_Tr     [  <=>               ] 395.87K  1.50MB/s    in 0.3s    

2026-03-29 10:06:49 (1.50 MB/s) - ‘Metro_Interstate_Traffic_Volume.csv.gz.1’ saved [405373]



In [ ]:
# Upload the data
df = pd.read_csv('Metro_Interstate_Traffic_Volume.csv.gz')

#STEP 2: Data Understanding + Cleaning

**1. Basic informations about data**

In [ ]:
# show first 5 rows
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [ ]:
# check the shape of dataset
df.shape

(48204, 9)

In [ ]:
# Info about data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     object 
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  object 
 6   weather_description  48204 non-null  object 
 7   date_time            48204 non-null  object 
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB


**2. Check Missing Values**

In [ ]:
df.isnull().sum()

,0
holiday,48143
temp,0
rain_1h,0
snow_1h,0
clouds_all,0
weather_main,0
weather_description,0
date_time,0
traffic_volume,0


**3. Convert Date Column (VERY IMPORTANT)**

In [ ]:
# we r doing this bcs the data in original df is in string data structure
df['date_time'] = pd.to_datetime(df['date_time'])

**4. Extract Useful Features**

In [ ]:
# the trafic folw/volume depends on these time intervals havily.
df['hour'] = df['date_time'].dt.hour
df['day'] = df['date_time'].dt.day_of_week
df['month'] = df['date_time'].dt.month

In [ ]:
df['day'].unique()

array([1, 2, 3, 4, 5, 6, 0], dtype=int32)

In [ ]:
df['month'].unique()

array([10, 11, 12,  1,  2,  3,  4,  5,  6,  7,  8,  9], dtype=int32)

**5. Create Target Variable (Classification)**

In [ ]:
# Lebeling the flow into three categories. It can be different for different countries.
def traffic_label(x):
    if x < 2000:
        return "Low"
    elif x < 4000:
        return "Medium"
    else:
        return "High"

df['traffic_level'] = df['traffic_volume'].apply(traffic_label)

In [ ]:
df['traffic_level'].value_counts()

,count
traffic_level,
High,20939
Low,15219
Medium,12046


**6. Drop Unnecessary Columns**

In [ ]:
df = df.drop(['date_time', 'traffic_volume'], axis=1)

**7. Convert Categorical → Numeric**

In [ ]:
# Separate target first
y = df['traffic_level']
X = df.drop('traffic_level', axis=1)

# Encode only features
# We r encoding becouse model can not utilise object values.
X = pd.get_dummies(X, drop_first=True)

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 64 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   temp                                                     48204 non-null  float64
 1   rain_1h                                                  48204 non-null  float64
 2   snow_1h                                                  48204 non-null  float64
 3   clouds_all                                               48204 non-null  int64  
 4   hour                                                     48204 non-null  int32  
 5   day                                                      48204 non-null  int32  
 6   month                                                    48204 non-null  int32  
 7   holiday_Columbus Day                                     48204 non-null  bool   
 8   holiday_Independence Day  

**8.Split the dataset**

In [ ]:
from sklearn.model_selection import train_test_split

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(38563, 64) (9641, 64)


**9.Scale features**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#STEP 3: Train First Model (Logistic Regression)

**1. Import Model**

In [ ]:
from sklearn.linear_model import LogisticRegression

**2. Create Model**

In [ ]:
model = LogisticRegression(max_iter=1000)

**3. Train Model**

In [ ]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

**4. Prediction**

In [ ]:
y_pred = model.predict(X_test)

**4.Classification Report**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print(classification_report(y_test, y_pred))
print("Accuracy:",accuracy_score(y_pred,y_test))

              precision    recall  f1-score   support

        High       0.72      0.81      0.76      4241
         Low       0.72      0.76      0.74      3032
      Medium       0.57      0.41      0.47      2368

    accuracy                           0.69      9641
   macro avg       0.67      0.66      0.66      9641
weighted avg       0.68      0.69      0.68      9641

Accuracy: 0.6938076962970646


#STEP 4: Use Random Forest (Better Model)

**1. Import Model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

**2. Train Model**

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

**3. Predict**

In [ ]:
y_pred_rf = rf_model.predict(X_test)

**4. Evaluate**

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print(classification_report(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

              precision    recall  f1-score   support

        High       0.93      0.96      0.95      4241
         Low       0.96      0.93      0.94      3032
      Medium       0.85      0.84      0.84      2368

    accuracy                           0.92      9641
   macro avg       0.91      0.91      0.91      9641
weighted avg       0.92      0.92      0.92      9641

Accuracy: 0.9198215952702001


**5.Feature importaince**

In [ ]:
import pandas as pd

importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(10))

hour                                    0.680585
temp                                    0.140681
day                                     0.078129
month                                   0.042201
clouds_all                              0.024577
rain_1h                                 0.008652
weather_main_Clouds                     0.002995
weather_description_sky is clear        0.001829
weather_main_Mist                       0.001408
weather_description_scattered clouds    0.001204
dtype: float64


#STEP 5: Hyperparameter Tuning (Random Forest)

**Method 1: GridSearch (BEST for learning)**

In [ ]:
# import
from sklearn.model_selection import GridSearchCV

**2. Define Parameter Grid**

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

**3.Apply GridSearch**

In [ ]:
rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


GridSearchCV(cv=3, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 20, None],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [50, 100]},
             verbose=2)

**4. Best Parameters**

In [ ]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}


**5. Use Best Model**

In [ ]:
best_rf = grid_search.best_estimator_

y_pred = best_rf.predict(X_test)

**6. Evaluate**

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))
print("Accuracy", accuracy_score(y_test,y_pred))

              precision    recall  f1-score   support

        High       0.93      0.97      0.95      4241
         Low       0.96      0.93      0.95      3032
      Medium       0.86      0.83      0.85      2368

    accuracy                           0.92      9641
   macro avg       0.92      0.91      0.91      9641
weighted avg       0.92      0.92      0.92      9641

Accuracy 0.9225184109532206


**Feature importance**

In [ ]:
import pandas as pd

importance = pd.Series(best_rf.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(10))

hour                                    0.713591
temp                                    0.111921
day                                     0.081711
month                                   0.036488
clouds_all                              0.023140
rain_1h                                 0.007470
weather_main_Clouds                     0.003211
weather_description_sky is clear        0.001872
weather_main_Mist                       0.001540
weather_description_scattered clouds    0.001319
dtype: float64


#BUILDING STREEAMLIT APP

**STEP 1: Build Optimized Model (Using Top Features)**

In [ ]:
# use most important/contributing features

features = ['hour', 'temp', 'day', 'month', 'clouds_all']

X = df[features]
y = df['traffic_level']

In [ ]:
# split data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train rf with best params

from sklearn.ensemble import RandomForestClassifier

best_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=1,
    random_state=42
)

best_rf.fit(X_train, y_train)

RandomForestClassifier(min_samples_split=5, random_state=42)

In [ ]:
# Evaluate

from sklearn.metrics import classification_report

y_pred = best_rf.predict(X_test)

print(classification_report(y_test, y_pred))
print("accuracy", accuracy_score(y_pred,y_test))

              precision    recall  f1-score   support

        High       0.95      0.98      0.96      4241
         Low       0.97      0.96      0.97      3032
      Medium       0.92      0.88      0.90      2368

    accuracy                           0.95      9641
   macro avg       0.95      0.94      0.94      9641
weighted avg       0.95      0.95      0.95      9641

accuracy 0.9479307125816824


In [ ]:
# Save the model

import pickle

pickle.dump(best_rf, open('traffic_model.pkl', 'wb'))